# 🏥 Heal Your Heart EHR — Remote Ollama GPU Host (qwen2.5:7b)

This notebook runs **Ollama** and the verified **`qwen2.5:7b`** model on a **Free Google Colab T4 GPU**.

It creates a secure public HTTPS link via Cloudflare Tunnel so developers with low-spec laptops can develop the EHR AI Agent without GPU hardware.

### ⚡ 3-Step Setup:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. Click the **Play (Run)** button on the cell below.
3. Copy the printed `OLLAMA_BASE_URL` into your local `backend/.env` file.

In [ ]:
# Step 1: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Step 2: Start Ollama service in background with GPU support
import subprocess, time, re, os, urllib.request

env = {**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"}
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env)

# Wait for Ollama to become ready
print("Waiting for Ollama service to start...")
for _ in range(15):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version")
        print("✅ Ollama service is up and running!")
        break
    except Exception:
        time.sleep(1)

# Step 3: Pull verified EHR model (qwen2.5:7b)
print("\n📥 Downloading qwen2.5:7b into Colab GPU memory (takes ~1 minute)...")
!ollama pull qwen2.5:7b

# Step 4: Install Cloudflare Tunnel (cloudflared - free, no signup required)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Step 5: Start Tunnel and print connection link for .env
print("\n🚀 Starting Cloudflare Tunnel... Generating public connection link...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

for line in iter(tunnel_proc.stderr.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print("\n" + "="*65)
        print("🎉 HEAL YOUR HEART — REMOTE OLLAMA GPU HOST IS ONLINE!")
        print("="*65)
        print("\n👉 Copy & paste this into your local 'backend/.env' file:\n")
        print(f"OLLAMA_BASE_URL={public_url}")
        print("OLLAMA_MODEL=qwen2.5:7b")
        print("OLLAMA_TIMEOUT=30.0\n")
        print("="*65)
        print("Keep this Colab tab running while developing your EHR.")
        print("="*65)
        break

# Keep process alive
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    tunnel_proc.terminate()
    ollama_proc.terminate()
    print("Server stopped.")

### 🧪 Optional: Quick Health Check Cell
Run this cell below to verify that `qwen2.5:7b` is loaded and ready in Colab.

In [ ]:
# Verify loaded models and version
!ollama list
!curl http://127.0.0.1:11434/api/version